In [19]:
!pip install -q keras-tuner

import keras_tuner as kt
import tensorflow as tf
import numpy as np
from tensorflow.keras import layers, models, regularizers, initializers
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [20]:
import pandas as pd
import joblib
X_test_processed = pd.read_csv('/content/drive/MyDrive/Customer-Churn-Prediction-ANN/data/proccessed/X_test_processed.csv')
X_train_processed = pd.read_csv('/content/drive/MyDrive/Customer-Churn-Prediction-ANN/data/proccessed/X_train_processed.csv')
y_test_processed = pd.read_csv('/content/drive/MyDrive/Customer-Churn-Prediction-ANN/data/proccessed/y_test_processed.csv')
y_train_processed = pd.read_csv('/content/drive/MyDrive/Customer-Churn-Prediction-ANN/data/proccessed/y_train_processed.csv')

label_encoder = joblib.load('/content/drive/MyDrive/Customer-Churn-Prediction-ANN/models/label_encoder.pkl')
scaler = joblib.load('/content/drive/MyDrive/Customer-Churn-Prediction-ANN/models/scaler.pkl')

In [21]:
print(X_test_processed.shape)


(2000, 11)


In [22]:
print(X_train_processed.shape)

(8000, 11)


In [23]:
# Training partition ke andar se hi ek validation split banate hain tuning ke liye
# Test set (X_test/y_test) is process mein bilkul touch nahi hoga
X_tr_tune, X_val_tune, y_tr_tune, y_val_tune = train_test_split(
    X_train_processed, y_train_processed, test_size=0.2, stratify=y_train_processed, random_state=SEED
)

print("Tuning train shape:", X_tr_tune.shape)
print("Tuning val shape:", X_val_tune.shape)

Tuning train shape: (6400, 11)
Tuning val shape: (1600, 11)


In [24]:
def build_model_for_tuning(hp):
    model = models.Sequential()
    model.add(layers.Input(shape=(X_train_processed.shape[1],)))

    # Number of hidden layers: 2 to 4
    n_layers = hp.Int('n_layers', min_value=2, max_value=4)

    for i in range(n_layers):
        units = hp.Choice(f'units_{i}', values=[16, 32, 64, 128])
        model.add(layers.Dense(
            units,
            activation='relu',
            kernel_initializer=initializers.HeNormal(seed=SEED),
            kernel_regularizer=regularizers.l2(
                hp.Choice(f'l2_{i}', values=[0.0001, 0.001, 0.01])
            )
        ))
        model.add(layers.BatchNormalization())

        dropout_rate = hp.Float(f'dropout_{i}', min_value=0.1, max_value=0.5, step=0.1)
        model.add(layers.Dropout(dropout_rate))

    model.add(layers.Dense(1, activation='sigmoid'))

    learning_rate = hp.Choice('learning_rate', values=[0.01, 0.001, 0.0005, 0.0001])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss='binary_crossentropy',
        metrics=['accuracy',
                 tf.keras.metrics.Precision(name='precision'),
                 tf.keras.metrics.Recall(name='recall'),
                 tf.keras.metrics.AUC(name='auc')]
    )
    return model

In [25]:
tuner = kt.BayesianOptimization(
    build_model_for_tuning,
    objective=kt.Objective('val_auc', direction='max'),   # AUC pe optimize, accuracy pe nahi (imbalance ki wajah se)
    max_trials=30,
    executions_per_trial=1,
    directory='tuner_results',
    project_name='churn_ann_tuning',
    overwrite=True,
    seed=SEED
)

tuner.search_space_summary()

Search space summary
Default search space size: 8
n_layers (Int)
{'default': None, 'conditions': [], 'min_value': 2, 'max_value': 4, 'step': 1, 'sampling': 'linear'}
units_0 (Choice)
{'default': 16, 'conditions': [], 'values': [16, 32, 64, 128], 'ordered': True}
l2_0 (Choice)
{'default': 0.0001, 'conditions': [], 'values': [0.0001, 0.001, 0.01], 'ordered': True}
dropout_0 (Float)
{'default': 0.1, 'conditions': [], 'min_value': 0.1, 'max_value': 0.5, 'step': 0.1, 'sampling': 'linear'}
units_1 (Choice)
{'default': 16, 'conditions': [], 'values': [16, 32, 64, 128], 'ordered': True}
l2_1 (Choice)
{'default': 0.0001, 'conditions': [], 'values': [0.0001, 0.001, 0.01], 'ordered': True}
dropout_1 (Float)
{'default': 0.1, 'conditions': [], 'min_value': 0.1, 'max_value': 0.5, 'step': 0.1, 'sampling': 'linear'}
learning_rate (Choice)
{'default': 0.01, 'conditions': [], 'values': [0.01, 0.001, 0.0005, 0.0001], 'ordered': True}


In [26]:
class_weights = compute_class_weight('balanced', classes=np.unique(y_tr_tune['Exited']), y=y_tr_tune['Exited'])
class_weight_dict = {i: w for i, w in enumerate(class_weights)}

early_stop_tune = tf.keras.callbacks.EarlyStopping(
    monitor='val_auc', mode='max', patience=10, restore_best_weights=True
)

tuner.search(
    X_tr_tune, y_tr_tune,
    validation_data=(X_val_tune, y_val_tune),
    epochs=80,
    batch_size=32,
    class_weight=class_weight_dict,
    callbacks=[early_stop_tune],
    verbose=1
)

Trial 30 Complete [00h 01m 29s]
val_auc: 0.8425542712211609

Best val_auc So Far: 0.8620787262916565
Total elapsed time: 00h 25m 34s


In [27]:
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

print("=== Best Hyperparameters Found ===")
for param in best_hps.values:
    print(f"{param}: {best_hps.values[param]}")

=== Best Hyperparameters Found ===
n_layers: 2
units_0: 32
l2_0: 0.001
dropout_0: 0.2
units_1: 128
l2_1: 0.0001
dropout_1: 0.1
learning_rate: 0.01
units_2: 32
l2_2: 0.001
dropout_2: 0.1
units_3: 16
l2_3: 0.001
dropout_3: 0.5


In [29]:
best_model = tuner.hypermodel.build(best_hps)

class_weights_full_train = compute_class_weight('balanced', classes=np.unique(y_train_processed['Exited']), y=y_train_processed['Exited'])
class_weight_dict_full = {i: w for i, w in enumerate(class_weights_full_train)}

early_stop_final = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=15, restore_best_weights=True
)
reduce_lr_final = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6
)

history_tuned = best_model.fit(
    X_train_processed, y_train_processed,
    validation_split=0.2,
    epochs=150,
    batch_size=32,
    class_weight=class_weight_dict_full,
    callbacks=[early_stop_final, reduce_lr_final],
    verbose=1
)

Epoch 1/150
200/200 ━━━━━━━━━━━━━━━━━━━━ 10s 8ms/step - accuracy: 0.7058 - auc: 0.7624 - loss: 0.6868 - precision: 0.3807 - recall: 0.6977 - val_accuracy: 0.7287 - val_auc: 0.8221 - val_loss: 0.5963 - val_precision: 0.4066 - val_recall: 0.7750 - learning_rate: 0.0100
Epoch 2/150
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7361 - auc: 0.8054 - loss: 0.6064 - precision: 0.4163 - recall: 0.7198 - val_accuracy: 0.7600 - val_auc: 0.8302 - val_loss: 0.5494 - val_precision: 0.4403 - val_recall: 0.7375 - learning_rate: 0.0100
Epoch 3/150
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7542 - auc: 0.8251 - loss: 0.5710 - precision: 0.4406 - recall: 0.7443 - val_accuracy: 0.7925 - val_auc: 0.8524 - val_loss: 0.4874 - val_precision: 0.4878 - val_recall: 0.7469 - learning_rate: 0.0100
Epoch 4/150
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7598 - auc: 0.8299 - loss: 0.5560 - precision: 0.4481 - recall: 0.7481 - val_accuracy: 0.7919 - val_auc: 0.8467 - val_loss: 0.4892 

In [31]:
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, classification_report

y_pred_prob_tuned = best_model.predict(X_test_processed).ravel()
y_pred_tuned_default = (y_pred_prob_tuned >= 0.5).astype(int)

print("=== Tuned Model (threshold=0.5) ===")
print(classification_report(y_test_processed, y_pred_tuned_default))
print("ROC-AUC:", roc_auc_score(y_test_processed, y_pred_prob_tuned))

# Comparison with previous best (class-weighted, Notebook 06)
comparison_final = pd.DataFrame([
    {'Model': 'Class-Weighted (manual arch)', 'Recall': 0.769, 'Precision': 0.472, 'ROC-AUC': 0.854},
    {'Model': 'Tuned (KerasTuner)',
     'Recall': recall_score(y_test_processed, y_pred_tuned_default),
     'Precision': precision_score(y_test_processed, y_pred_tuned_default),
     'ROC-AUC': roc_auc_score(y_test_processed, y_pred_prob_tuned)}
])
print(comparison_final)
comparison_final.to_csv('tuning_comparison.csv', index=False)

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
=== Tuned Model (threshold=0.5) ===
              precision    recall  f1-score   support

           0       0.91      0.83      0.87      1593
           1       0.51      0.70      0.59       407

    accuracy                           0.80      2000
   macro avg       0.71      0.76      0.73      2000
weighted avg       0.83      0.80      0.81      2000

ROC-AUC: 0.853698074037057
                          Model    Recall  Precision   ROC-AUC
0  Class-Weighted (manual arch)  0.769000    0.47200  0.854000
1            Tuned (KerasTuner)  0.695332    0.50991  0.853698
